## Loading the Dataset, Noise Reduction, and Sanitization

In [1]:
import numpy as np
import pandas as pd

#  Load a safe slice (80,000 rows) to protect RAM
df = pd.read_csv("friday.csv", nrows=80000)

#  Delete noise/metadata columns
noise_cols = ['id', 'Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Src Port', 'Dst Port', 'Attempted Category']
existing_noise_cols = [col for col in noise_cols if col in df.columns]
df.drop(columns=existing_noise_cols, inplace=True)


df.head()

,Protocol,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label
0,6,112740690,32,16,6448,1152,403,0,201.50,204.724205,...,380,343,1.610540e+07,4.988048e+05,16399772,15375229,-1,-1,112740690,BENIGN
1,6,112740560,32,16,6448,5056,403,0,201.50,204.724205,...,330,285,1.610543e+07,4.987937e+05,16399782,15375263,-1,-1,112740560,BENIGN
2,0,113757377,545,0,0,0,0,0,0.00,0.000000,...,18851791,19,1.221036e+07,6.935824e+06,20757030,5504997,-1,-1,0,BENIGN
3,17,91997219,388,0,37151,0,227,37,95.75,55.785320,...,24721964,16,1.319764e+07,5.826905e+06,19776791,5817470,-1,-1,0,BENIGN
4,17,66966070,6,6,288,288,48,48,48.00,0.000000,...,1968172,1968172,6.497443e+07,0.000000e+00,64974431,64974431,-1,-1,0,BENIGN


In [2]:
# Replace inf and -inf with NaN first to handle them uniformly
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaN values with the median of each column or drop them
initial_rows = df.shape[0]
df.dropna(inplace=True)
final_rows = df.shape[0]


### Translation and Signal Scaling

In [3]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

#Separate features (X) and label (y)
y_raw = df['Label']
X_raw = df.drop(columns=['Label'])

# Binary conversion of the label: BENIGN -> 0, any attack -> 1
y = np.where(y_raw == 'BENIGN', 0, 1)


# Identify any object columns left in X and convert them using one-hot/label encoding if necessary
non_numeric_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print(f"Found non-numeric features to translate: {non_numeric_cols}")
    X_raw = pd.get_dummies(X_raw, columns=non_numeric_cols)
else:
    print("All features are already numeric.")

# Voltage Equalization (MinMax Scaling to [0, 1])
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

All features are already numeric.


### 3D Time-Series Shaping (The Core LSTM Prep)

In [4]:
def create_sequences(features, targets, window_size=10):
    X_seq = []
    y_seq = []
    # Iterate through the data to build sliding windows
    for i in range(len(features) - window_size + 1):
        X_seq.append(features[i:i + window_size])
        # Use the target of the final packet in the sequence window as the label
        y_seq.append(targets[i + window_size - 1])
    return np.array(X_seq), np.array(y_seq)

# Configure window size of 10 packets
WINDOW_SIZE = 10

X_3d, y_3d = create_sequences(X_scaled, y, window_size=WINDOW_SIZE)

print(f"Reshaped 3D feature shape (Samples, Sequence Length, Features): {X_3d.shape}", f"Reshaped label shape: {y_3d.shape}")


Reshaped 3D feature shape (Samples, Sequence Length, Features): (79991, 10, 82) Reshaped label shape: (79991,)


### LSTM Network Construction and Training with Safety Nets

In [5]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


X_train, X_test, y_train, y_test = train_test_split(
    X_3d, y_3d, test_size=0.2, random_state=42, stratify=y_3d
)



# Build the Network with LSTM & Dropout layers
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model with binary classification metrics
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 3. Train the Model
print("\nBeginning training sequence...")
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, y_test),
    verbose=1
)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        37,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,745 (155.25 KB)

 Trainable params: 39,745 (155.25 KB)

 Non-trainable params: 0 (0.00 B)


Beginning training sequence...
Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - accuracy: 0.9841 - loss: 0.0599 - val_accuracy: 0.9954 - val_loss: 0.0149
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.9961 - loss: 0.0135 - val_accuracy: 0.9958 - val_loss: 0.0141
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 30ms/step - accuracy: 0.9963 - loss: 0.0129 - val_accuracy: 0.9957 - val_loss: 0.0131
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9964 - loss: 0.0124 - val_accuracy: 0.9958 - val_loss: 0.0126
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 36ms/step - accuracy: 0.9964 - loss: 0.0115 - val_accuracy: 0.9958 - val_loss: 0.0125


### Model Evaluation 

In [6]:
from sklearn.metrics import classification_report, confusion_matrix
 # Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Make predictions and convert probabilities to binary classes
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Print metrics summary (Note: Monday dataset is typically entirely benign traffic)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Test Loss: 0.0125
Test Accuracy: 99.58%
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     15464
           1       0.89      0.99      0.94       535

    accuracy                           1.00     15999
   macro avg       0.95      1.00      0.97     15999
weighted avg       1.00      1.00      1.00     15999

